<a href="https://colab.research.google.com/github/AbdourahmaneFatmaAli/lab-4-llm-decision-support/blob/main/lab_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [ ]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash-lite"
print("client ready.")

client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

In [ ]:
response = ask_llm_with_retry("Say hello in one sentence.", temperature=0)
print(response.text)

Hello there, I hope you are having a wonderful day!


In [ ]:
import time

def ask_llm_with_retry(question, temperature, max_tokens=1500, retries=3):
    for attempt in range(retries):
        try:
            return ask_llm(question, temperature=temperature, max_tokens=max_tokens)
        except Exception as e:
            print(f"Attempt {attempt+1} failed ({e}). Retrying in 15s...")
            time.sleep(15)
    raise Exception("Failed after multiple retries")


question = "Suggest a name for a savings product for market traders in Accra."

print("=== Temperature = 0.0 ===\n")
temp0_answers = []
for i in range(5):
    response = ask_llm_with_retry(question, temperature=0.0)
    temp0_answers.append(response.text)
    print(f"Run {i+1}: {response.text}\n")
    time.sleep(5)

print("\n=== Temperature = 1.2 ===\n")
temp12_answers = []
for i in range(5):
    response = ask_llm_with_retry(question, temperature=1.2)
    temp12_answers.append(response.text)
    print(f"Run {i+1}: {response.text}\n")
    time.sleep(5)

=== Temperature = 0.0 ===

Run 1: Here are several name suggestions for a savings product tailored for market traders in Accra, broken down by the "vibe" or branding angle you might want to take:

### 1. Local & Cultural Resonance (Ga/Twi Mix)
*These names use familiar local concepts of wealth creation, daily collection, and community trust.*

*   **Sika Dwa Savings** (*"Sika"* = Money/Wealth, *"Dwa"* = Market/Trading seat in Twi). Simple, prestigious, and directly ties savings to market business.
*   **Korkor’sika Box** (Inspired by the Ga name Korkor and *Sika*). Feels personal, like a trusted friend helping you save.
*   **Adzato Daily** (*"Adzato"* means turning things around or making progress in Ga). Great for daily contribution (susu) models.
*   **Nkabom Trader Account** (*"Nkabom"* = Unity/Coming together in Twi). Appeals to market association groups (Market Queens, union members).

### 2. Trust, Security & Growth (Pidgin & English blend)
*Accra traders value security against 

### Part 1.1 — Your first API call

In [ ]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
# def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
#             temperature=0.7, max_tokens=500):
#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": system_prompt},
#             {"role": "user",   "content": user_prompt},
#         ],
#         temperature=temperature,
#         max_tokens=max_tokens,
#     )
#     return response.choices[0].message.content
#
# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?

**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> **Answer:**


System = the instructions that set the model's role and rules for the whole task. Example You are an assistant to a loan officer,Don't invent details.
User = the actual question or input for this specific call

A token is roughly a piece of a word.

Providers bill per token because the cost depends on how much text the model reads and generates and not on how many times you call the API

### Part 1.2 — Temperature: the randomness dial

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.

In [ ]:
import time

def ask_llm_with_retry(question, temperature, max_tokens=1500, retries=3):
    for attempt in range(retries):
        try:
            return ask_llm(question, temperature=temperature, max_tokens=max_tokens)
        except Exception as e:
            print(f"Attempt {attempt+1} failed ({e}). Retrying in 15s...")
            time.sleep(15)
    raise Exception("Failed after multiple retries")


question = "Suggest a name for a savings product for market traders in Accra."

print("=== Temperature = 0.0 ===\n")
temp0_answers = []
for i in range(5):
    response = ask_llm_with_retry(question, temperature=0.0)
    temp0_answers.append(response.text)
    print(f"Run {i+1}: {response.text}\n")
    time.sleep(5)

print("\n=== Temperature = 1.2 ===\n")
temp12_answers = []
for i in range(5):
    response = ask_llm_with_retry(question, temperature=1.2)
    temp12_answers.append(response.text)
    print(f"Run {i+1}: {response.text}\n")
    time.sleep(5)

=== Temperature = 0.0 ===

Run 1: Naming a savings product for market traders in Accra requires a blend of cultural resonance, trust, aspiration, and practicality. Traders in iconic markets like Makola, Kejetia (though Kumasi, the vibe is similar), Kaneshie, or Agbogbloshie value growth, security, and easy access to their money. 

Here are several name suggestions categorized by the "vibe" they project, along with the reasoning behind them:

### 1. High Energy & Aspirational (Focus on growth and success)
*   **Sika Dwa Plus** (*Sika Dwa* means "Golden Stool" or "Money Chair" in Twi, symbolizing wealth and authority). 
    *   *Why it works:* It connects modern savings with traditional Ghanaian prestige. 
*   **Adepa Nest Egg** (*Adepa* means "good thing" or "valuable thing" in Twi).
    *   *Why it works:* It sounds reliable and promises a secure future for their business or family.
*   **Kakra Kakra Build** (Inspired by the Twi proverb *“Kakra kakra adzekye”* – little by little makes 

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

> **Answer:** [Double-click to edit]

At temperature=0.0, the answers were still different each time even though the structure stayed the same. At temperature=1.2, the answers were also varied, but not much wilder than temp=0.



I would use temperature=0. The loan system needs to give the same, factual answer every time for the same letter. A loan officer should not get a different summary or different numbers just by chance. High temperature is fine for creative tasks like naming, but not for a system that needs to be consistent and trustworthy with real financial decisions.

---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [ ]:

SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    response = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}", temperature=0)
    print(f"--- V1 Summary: {letter_id} ---")
    print(response.text)
    print()

--- V1 Summary: L002 ---
**Summary:**

Kwame Boateng, a commercial driver from Kumasi, is urgently requesting a loan of GHS 25,000 to repair his trotro engine and pay off personal debts. He has no collateral and cannot provide a specific repayment date, stating only that he will pay back when business picks up after the festive season.

--- V1 Summary: L006 ---
**Summary:**

Kofi, a 22-year-old, is seeking a GHS 50,000 loan with no collateral to start three different ventures: a car washing business, a provision shop, and a phone importation business from Dubai. He has no prior business experience, relies on friends' opinions for validation, and promises to repay the loan in one year based on anticipated business success, assuring the lender of his trustworthiness.



In [ ]:

SUMMARY_SYSTEM_PROMPT = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan applications factually and neutrally. "
    "Do not invent details that are not stated in the letter. "
    "Do not give opinions or recommendations. "
    "Keep the summary to 3-4 sentences."
)

def SUMMARY_PROMPT_V2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    response = ask_llm(
        SUMMARY_PROMPT_V2(letter_text),
        system_prompt=SUMMARY_SYSTEM_PROMPT,
        temperature=0
    )
    print(f"--- V2 Summary: {letter_id} ---")
    print(response.text)
    print()

--- V2 Summary: L002 ---
Kwame Boateng, a commercial driver based in Kumasi, is applying for a loan of GHS 25,000. The requested funds are intended to repair his trotro engine and settle personal debts. He states that business has been slow but expects it to improve after the festive season. The applicant currently has no collateral to offer and proposes to repay the loan whenever funds become available.

--- V2 Summary: L006 ---
Kofi, a 22-year-old applicant, is requesting a loan of GHS 50,000 to start a car washing business, open a provision shop, and import phones from Dubai. He has not yet started any of these proposed businesses. He plans to repay the loan in one year once his businesses are booming. The application states that he has no collateral to offer, but he considers himself trustworthy and relies on friends' assessments of his business mindset.



In [ ]:
print("=" * 60)
print("V1 vs V2 COMPARISON")
print("=" * 60)

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]

    v1_response = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}", temperature=0)
    v2_response = ask_llm(
        SUMMARY_PROMPT_V2(letter_text),
        system_prompt=SUMMARY_SYSTEM_PROMPT,
        temperature=0
    )

    print(f"\n### {letter_id} ###")
    print(f"\n[V1]\n{v1_response.text}")
    print(f"\n[V2]\n{v2_response.text}")
    print("-" * 60)

V1 vs V2 COMPARISON

### L002 ###

[V1]
**Summary:**

Kwame Boateng, a commercial driver in Kumasi, is urgently requesting a GHS 25,000 loan with no collateral to repair his trotro engine and pay personal debts. He expects business to improve after the festive season and promises to repay whenever funds are available.

[V2]
Kwame Boateng, a commercial driver from Kumasi, is applying for an urgent loan of GHS 25,000. The funds are needed to repair his trotro engine and settle personal debts. He anticipates business will improve after the festive season and proposes to repay the loan whenever the money comes. The applicant currently has no collateral to offer.
------------------------------------------------------------

### L006 ###

[V1]
**Summary:**

Kofi, a 22-year-old, is applying for a GHS 50,000 loan to simultaneously start three different ventures: a car wash, a provision shop, and a Dubai phone importation business. He currently has no business experience or collateral, relies o

In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> **Answer:**


V1 added a bold Summary: header that wasn't asked for and extra formatting, not neutral. V1 also sounded more editorial in places, like V1's L002 said the applicant cannot provide a specific repayment date,which is Claude's own judgment added on top of the facts, not something stated directly by the applicant. V2 stuck closer to just restating facts in the letter, without a header and without added commentary.

V1 was also inconsistent in style between the two letters one used a header, the other was more narrative. V2 was consistent same neutral tone and structure both times, because the system prompt fixed a role and rules for every call.

2. Why is no invented details essential? What is this failure mode called?

In a loan decision system, invented details could make an applicant look more or less creditworthy than they really are, a false detail could lead to a wrong decision. This failure mode is called hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [ ]:
import json
import pandas as pd

EXTRACT_SYSTEM_PROMPT = """You are a data extraction assistant for a microfinance loan officer.
You must extract information from loan application letters and return ONLY a JSON object
with EXACTLY these keys:

- applicant_name (string)
- amount_ghs (number)
- purpose (string)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean)
- repayment_months (number or null)

Rules:
- If a field is not stated in the letter, use null. Do not guess.
- Return ONLY the JSON object. No extra text, no explanation, no markdown fences.

Example letter:
"Dear Sir, my name is Ama Serwaa. I sell vegetables at Madina Market. I need GHS 3,000
to buy a wheelbarrow and expand my delivery service. My monthly profit is about GHS 600.
My brother will act as guarantor. I can repay over 10 months."

Example output:
{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 3000,
  "purpose": "buy a wheelbarrow and expand delivery service",
  "monthly_profit_ghs": 600,
  "has_collateral_or_guarantor": true,
  "repayment_months": 10
}
"""

def EXTRACT_PROMPT(letter_text):
    return f"Extract the fields from this loan application letter:\n\n{letter_text}"


def extract_fields(letter_text):
    response = ask_llm_with_retry(
        EXTRACT_PROMPT(letter_text),
        temperature=0.0
    )
    raw_text = response.text.strip()

    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        raw_text = raw_text.replace("json\n", "", 1).replace("json", "", 1)
        raw_text = raw_text.strip()

    try:
        return json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"Warning: failed to parse JSON. Error: {e}")
        print(f"Raw output was:\n{raw_text}")
        return None

In [ ]:
def ask_llm_with_retry(question, temperature, system_prompt="You are a helpful assistant.", max_tokens=1500, retries=3):
    for attempt in range(retries):
        try:
            return ask_llm(question, system_prompt=system_prompt, temperature=temperature, max_tokens=max_tokens)
        except Exception as e:
            print(f"Attempt {attempt+1} failed ({e}). Retrying in 15s...")
            time.sleep(15)
    raise Exception("Failed after multiple retries")


def extract_fields(letter_text):
    response = ask_llm_with_retry(
        EXTRACT_PROMPT(letter_text),
        system_prompt=EXTRACT_SYSTEM_PROMPT,
        temperature=0.0
    )
    raw_text = response.text.strip()

    if raw_text.startswith("```"):
        raw_text = raw_text.strip("`")
        raw_text = raw_text.replace("json\n", "", 1).replace("json", "", 1)
        raw_text = raw_text.strip()

    try:
        return json.loads(raw_text)
    except json.JSONDecodeError as e:
        print(f"Warning: failed to parse JSON. Error: {e}")
        print(f"Raw output was:\n{raw_text}")
        return None

In [ ]:
results = []
for letter_id, letter_text in LETTERS.items():
    print(f"Extracting {letter_id}...")
    fields = extract_fields(letter_text)
    if fields is not None:
        fields["letter_id"] = letter_id
        results.append(fields)
    time.sleep(5)

df_extracted = pd.DataFrame(results)
cols = ["letter_id"] + [c for c in df_extracted.columns if c != "letter_id"]
df_extracted = df_extracted[cols]

df_extracted

Extracting L001...
Extracting L002...
Extracting L003...
Extracting L004...
Extracting L005...
Extracting L006...


,letter_id,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
0,L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
1,L002,Kwame Boateng,25000,repair my trotro engine and settle some person...,NaN,False,NaN
2,L003,Efua Darko,15000,purchase two industrial sewing machines and fa...,2800.0,True,15.0
3,L004,Yaw Owusu,12000,buy feed and 500 new layers for poultry farm,1500.0,True,18.0
4,L005,Adenta Women's Weaving Cooperative,30000,buy a bulk order of yarn directly from the fac...,NaN,True,16.0
5,L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


In [ ]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> **Answer:**

The example must not come from the six letters because the model could just copy that letter's answer instead of really learning the pattern. It would make results look better than they really are.

Without use null, do not guess, the model tends to make up a value when information is missing, instead of saying it's missing. This is called hallucination, and it's risky in a loan system.

Temperature=0 is right for extraction because there is one correct answer the facts in the letter, so we want the same answer every time. Creative tasks don't have one right answer, so a higher temperature helps the model give more varied, original ideas.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [ ]:
BRIEF_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer in Ghana.
You help prepare decision-support briefs for loan applications. You do NOT make the final
decision — the human loan officer always makes the final call to approve or reject.

Given a loan application letter and its extracted data, produce a brief with exactly these
four sections:

1. Strengths (bullet points, grounded only in facts from the letter)
2. Risks / Red flags (bullet points)
3. Missing information the officer should request
4. Suggested next step — choose ONE of: "invite for interview", "request documents",
   "flag for senior review". Do NOT say "approve" or "reject" — that decision belongs
   to the human loan officer, not you.

Be factual and neutral. Do not invent details not present in the letter or the extracted data.
"""

def BRIEF_PROMPT(letter_text, extracted_json):
    return f"""Loan application letter:
{letter_text}

Extracted data:
{extracted_json}

Prepare the decision-support brief."""

In [ ]:
def generate_brief(letter_text, extracted_json):
    response = ask_llm_with_retry(
        BRIEF_PROMPT(letter_text, extracted_json),
        system_prompt=BRIEF_SYSTEM_PROMPT,
        temperature=0.0
    )
    return response.text

In [ ]:
briefs = {}
for letter_id, letter_text in LETTERS.items():
    print(f"Generating brief for {letter_id}...")
    extracted = df_extracted[df_extracted["letter_id"] == letter_id].to_dict(orient="records")[0]
    brief_text = generate_brief(letter_text, extracted)
    briefs[letter_id] = brief_text
    time.sleep(5)

print("All briefs generated.")

Generating brief for L001...
Generating brief for L002...
Generating brief for L003...
Generating brief for L004...
Generating brief for L005...
Generating brief for L006...
All briefs generated.


In [ ]:
for letter_id in ["L001", "L002", "L006"]:

    print(f"BRIEF: {letter_id}")
    print(briefs[letter_id])

BRIEF: L001
**1. Strengths**
* 12 years of business experience selling provisions at Makola Market.
* Proven savings track record with the institution's susu scheme over the past two years with zero missed contributions, accumulating GHS 2,500 in savings.
* Existing business generates a monthly profit of approximately GHS 900.
* Has identified a guarantor (her sister, who is a teacher).
* Clear loan purpose (to purchase a deep freezer and expand into frozen foods).

**2. Risks / Red flags**
* The proposed monthly repayment of GHS 450 represents 50% of the current monthly profit of GHS 900, which may strain cash flow if existing business revenue fluctuates or if the expansion does not immediately increase income.
* The loan tenure is 20 months, which is relatively long for micro-inventories and equipment without verified historical demand for frozen foods.

**3. Missing information the officer should request**
* Detailed cost breakdown of the deep freezer and the initial stock of frozen

In [ ]:
print(briefs["L003"])

Here is the decision-support brief for your review:

1. Strengths
- Registered business in Takoradi (Darko Fashions, registration no. BN-2019-4482) with three employed apprentices.
- Clear loan purpose: to purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
- Demonstrated strong seasonal revenue, with December revenue alone reaching GHS 22,000 last year.
- Average monthly profit reported at GHS 2,800, which supports the proposed monthly repayment of GHS 1,100 over 15 months.
- Offers a fixed deposit of GHS 5,000 with GCB as collateral.
- Attached sales records for the past 18 months are available for verification.

2. Risks / Red flags
- The proposed 15-month repayment term extends past the immediate Christmas season, meaning future cash flow consistency outside of peak periods must be verified.
- Reliance on seasonal spikes (such as December revenue) to support overall business stability.

3. Missing information the officer should request
- Verifica

In [ ]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> **Answer:**


Yes, the system got it right for both. For L003 strong, it correctly listed real strengths like a registered business, proof of income, and collateral. It also flagged fair risks, like relying too much on Christmas season sales. For L006 weak, it correctly flagged that Kofi has no experience, no collateral, and no started business yet, and it didn't treat his friends' opinions as real proof. So the system told strong and weak applications apart correctly.

The model doesn't know the bank's full rules or the applicant's full history, so it could get a real decision wrong. Ethical reason loan decisions affect people's lives, so a human should be responsible for the final choice, not an AI, and someone needs to be accountable if the decision is questioned.


### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> **Commit hash:** [paste here]

In [ ]:
prompts_content = '''
SUMMARY_SYSTEM_PROMPT = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize loan applications factually and neutrally. "
    "Do not invent details that are not stated in the letter. "
    "Do not give opinions or recommendations. "
    "Keep the summary to 3-4 sentences."
)

def SUMMARY_PROMPT(letter_text):
    return f"Summarize this loan application:\\n\\n{letter_text}"


EXTRACT_SYSTEM_PROMPT = (
    "You are a data extraction assistant for a microfinance loan officer. "
    "Return ONLY a JSON object with keys: applicant_name, amount_ghs, purpose, "
    "monthly_profit_ghs, has_collateral_or_guarantor, repayment_months. "
    "If a field is not stated, use null. Do not guess."
)

def EXTRACT_PROMPT(letter_text):
    return f"Extract the fields from this loan application letter:\\n\\n{letter_text}"


BRIEF_SYSTEM_PROMPT = (
    "You are an assistant to a microfinance loan officer. You do NOT make the final "
    "decision. Produce a brief with: Strengths, Risks, Missing information, and a "
    "Suggested next step (never approve/reject)."
)

def BRIEF_PROMPT(letter_text, extracted_json):
    return f"Letter:\\n{letter_text}\\n\\nExtracted data:\\n{extracted_json}\\n\\nPrepare the brief."
'''

with open('prompts.py', 'w') as f:
    f.write(prompts_content)

print("prompts.py created.")

prompts.py created.


---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [ ]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.

### Part 4.2 — Reliability: is the system consistent?

In [ ]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.

### Part 4.3 — Hallucination probing

In [ ]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.

**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> **Answer:** [Double-click to edit]

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> **Answer:** [Double-click to edit]

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> **Answer:** [Double-click to edit]

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.